# E3 - benchmark against the current stability predictors

**Pre-registered in `docs/preregistration/E3.md` @ `ccd3446`.** Every threshold, every arm and the
handling of each structure come from that file.

### What carries a verdict, and what does not (Part 6.1)

| | role | why |
|---|---|---|
| **SaProtΔG** | verdict | 0 of our 15 domains in its published training split |
| **ESM3ΔG** | verdict | 0 of 15 |
| ThermoMPNN | **descriptive, no verdict at any n** | **13 of 15** are in its train (12) or val (1) split |
| ProteinDPO | dropped before any result | needs ESM-IF plus the authors' weights; cost and risk |

**Significance: p < 0.025**, which is 0.05 across the two competitors that carry a verdict.

The ThermoMPNN exclusion is the reason this benchmark is trustworthy rather than a leaderboard:
23,191 of our 27,590 held-out variants are in its training data, so running it across all 15
domains would have scored its own training set against our held-out set. It runs on **FYN and
NPHP1 only** (4,399 variants), the two whose MegaScale constructs appear in no ThermoMPNN split.

### The four experiments (Part 2)

| Exp | version | structures | seeds |
|---|---|---|---|
| **1 — MAIN** | **base** | **measured constructs** | average of 3 |
| 2 | augmented | measured constructs | average of 3 |
| 3 | base | AlphaFold, trimmed | average of 3 |
| 4 | base | measured constructs | each seed alone |

**Experiment 1 is the verdict, always.** No experiment is promoted after results are seen.

### Cost, and why this notebook is resumable

A saturation scan is ~1,200 forward passes per construct; 26 constructs × 3 seeds × 2 models is
roughly 190,000. That is hours, and a Colab session will not always survive it. **Every scan is
cached to Drive per (model, version, seed, construct)** and skipped if present, so re-running the
notebook resumes rather than restarts. Run Experiment 1 first; 2, 3 and 4 are behind flags and can
be done in later sessions.

**ESM3 is gated.** Accept the licence at `huggingface.co/EvolutionaryScale/esm3-sm-open-v1` and set
`HF_TOKEN` before section 3, or ESM3ΔG is recorded as *not run* and the other arms stand (Part 2).

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, hashlib, subprocess, shutil
from pathlib import Path
import pandas as pd, numpy as np
from scipy.stats import spearmanr, wilcoxon

DRIVE = Path('/content/drive/MyDrive')

def pick(*names, what=''):
    for n in names:
        if (DRIVE / n).exists(): return DRIVE / n
    raise FileNotFoundError(f'No {what} folder in MyDrive. Tried: ' + ', '.join(names))

PROJECT = pick('sh3_repro/extended', 'SH3_extended', 'SH3 extended', what='project')
FROZEN  = pick('sh3_repro', what='frozen-run')          # READ ONLY
FROZEN_IN = PROJECT / 'input' / 'frozen'
OUT   = PROJECT / 'results' / 'E3_sota_benchmark'
CACHE = OUT / 'cache'                                   # per-construct, so a timeout is cheap
STRUCT = PROJECT / 'input' / 'external' / 'structures'
for d in (OUT, CACHE, STRUCT): d.mkdir(parents=True, exist_ok=True)

def frozen(rel, bundled):
    p = FROZEN / 'results' / rel
    return p if p.exists() else FROZEN_IN / 'results' / bundled

ALPHA = 0.025                    # Part 6.1: 0.05 across the two competitors with a verdict
AAS   = 'ACDEFGHIKLMNPQRSTVWY'   # the alphabet BOTH competitors index their scans by
print('project:', PROJECT); print('frozen :', FROZEN)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader 2>/dev/null || print('no GPU')

## 1. The pinned inputs, and a check that our own numbers are the frozen ones

Nothing about our model is re-run. B1's held-out predictions are read as they were written, and
the per-domain correlations are recomputed from them and asserted against `folding_per_domain_ci.csv`.
If those disagree, the comparison is against something that is not our published model.

In [ ]:
H = pd.read_parquet(frozen('B1_folding_model/heldout_predictions.parquet',
                           'heldout_predictions.parquet'))
CI = pd.read_csv(frozen('B1_folding_model/folding_per_domain_ci.csv',
                        'folding_per_domain_ci.csv'))
CI = CI[CI.gene != 'gene'].copy()
for c in ('n', 'rho'): CI[c] = pd.to_numeric(CI[c])

print(f'{len(H):,} held-out variants | {H.gene.nunique()} domains | '
      f'{H.domain_key.nunique()} MegaScale constructs')
assert H.gene.nunique() == 15, 'expected the 15 held-out SH3 domains'

# our per-domain rho, recomputed from the frozen predictions
OURS = (H.groupby(['group', 'gene'])
         .apply(lambda g: pd.Series({'n': len(g),
                                     'rho_ours': spearmanr(g.pred, g.ddG_fold).statistic}),
                include_groups=False)
         .reset_index())
chk = OURS.merge(CI[['group', 'gene', 'n', 'rho']], on=['group', 'gene'], suffixes=('', '_rec'))
d = (chk.rho_ours - chk.rho).abs().max()
assert len(chk) == 15 and d < 1e-6, f'our per-domain rho does not reproduce B1 (max diff {d:.2e})'
print(f'our 15 per-domain rho reproduce B1 exactly (max diff {d:.1e}); '
      f'median {OURS.rho_ours.median():.3f}')

## 2. Constructs, structures, and the position mapping

Part 6.3. For MegaScale rows the frozen run set `domain_key` to the construct name and `pos` to the
1-based position **inside that construct**, so every variant already names its own structure. The
construct sequences come from A2's feature table, not from the 2.1 GB MegaScale file.

In [ ]:
MG = pd.read_parquet(frozen('A2_features/features_megascale.parquet',
                            'features_megascale.parquet'),
                     columns=['domain_key', 'pos', 'wt_aa', 'wt_seq'])
SEQ = (MG[MG.domain_key.isin(H.domain_key.unique())]
       .drop_duplicates('domain_key').set_index('domain_key').wt_seq.to_dict())
assert set(SEQ) == set(H.domain_key.unique()), 'a held-out construct has no sequence'

# every held-out variant must sit on its own construct sequence
bad = [(k, p, w) for k, p, w in zip(H.domain_key, H.pos, H.wt_aa)
       if not (1 <= p <= len(SEQ[k]) and SEQ[k][p-1] == w)]
assert not bad, f'{len(bad)} variants do not match their construct sequence, e.g. {bad[:3]}'
print(f'{len(SEQ)} constructs, all {len(H):,} variants sit on their own sequence')

PDB_OF = lambda k: k.split('.pdb')[0].upper()
BG     = lambda k: (re.search(r'\.pdb_([A-Z])(\d+)([A-Z])$', k) or [None])
entries = sorted({PDB_OF(k) for k in SEQ})
nbg = sum(1 for k in SEQ if re.search(r'\.pdb_[A-Z]\d+[A-Z]$', k))
print(f'{len(entries)} PDB entries: {" ".join(entries)}')
print(f'{nbg} of {len(SEQ)} constructs are background point mutants')
n_bg_var = int(H.domain_key.str.contains(r'\.pdb_[A-Z]\d+[A-Z]$', regex=True).sum())
print(f'  they carry {n_bg_var:,} of {len(H):,} held-out variants - parent coordinates are used '
      f'and the background substitution is applied to the sequence only')

In [ ]:
# fetch each entry once from RCSB; NMR entries keep model 1 only (Part 6.3)
try:
    from Bio.PDB import PDBParser, PDBIO, Select
except ImportError:
    !pip -q install biopython
    from Bio.PDB import PDBParser, PDBIO, Select

def fetch(pdb_id):
    dst = STRUCT / f'{pdb_id}.pdb'
    if dst.exists() and dst.stat().st_size > 0: return dst
    url = f'https://files.rcsb.org/download/{pdb_id}.pdb'
    r = subprocess.run(['curl', '-sL', '--fail', '-o', str(dst), url])
    assert r.returncode == 0 and dst.stat().st_size > 0, f'could not fetch {pdb_id}'
    return dst

class Model1(Select):
    def accept_model(self, m): return m.id == 0

parser = PDBParser(QUIET=True); io = PDBIO()
STRUCT_INFO = {}
for e in entries:
    src = fetch(e)
    st = parser.get_structure(e, src)
    n_models = len(st)
    one = STRUCT / f'{e}_model1.pdb'
    if not one.exists():
        io.set_structure(st); io.save(str(one), Model1())
    chains = [c.id for c in st[0]]
    STRUCT_INFO[e] = dict(path=one, n_models=n_models, chains=chains)
    print(f'{e}  models {n_models:>2}  chains {chains}')

In [ ]:
# pick the chain whose sequence matches the construct, and map construct pos -> chain index
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from Bio import Align

def chain_seq(path, chain_id):
    st = parser.get_structure('x', path)
    out = []
    for res in st[0][chain_id]:
        try: out.append((index_to_one(three_to_index(res.get_resname())), res.id[1]))
        except Exception: pass
    return ''.join(a for a, _ in out), [n for _, n in out]

al = Align.PairwiseAligner(mode='global', open_gap_score=-11, extend_gap_score=-1)
al.match_score, al.mismatch_score = 2, -1

MAPPING = {}       # construct -> dict(entry, chain, pdb_path, c2s: construct pos -> struct index)
rows = []
for k, seq in SEQ.items():
    e = PDB_OF(k); info = STRUCT_INFO[e]
    best = None
    for ch in info['chains']:
        s, _ = chain_seq(info['path'], ch)
        if len(s) < 20: continue
        sc = al.score(seq, s) / max(len(seq), 1)
        if best is None or sc > best[0]: best = (sc, ch, s)
    assert best is not None, f'no usable chain in {e}'
    _, ch, s = best
    a = al.align(seq, s)[0]
    c2s = {cs + i + 1: ss + i for (cs, ce), (ss, se) in zip(a.aligned[0], a.aligned[1])
           for i in range(ce - cs)}
    agree = sum(1 for p, j in c2s.items() if seq[p-1] == s[j])
    MAPPING[k] = dict(entry=e, chain=ch, path=info['path'], c2s=c2s, struct_seq=s)
    rows.append((k, e, ch, len(seq), len(s), len(c2s), round(agree / max(len(c2s), 1), 3)))
MAP = pd.DataFrame(rows, columns=['construct', 'entry', 'chain', 'len_construct',
                                  'len_chain', 'mapped', 'identity_on_mapped'])
MAP.to_csv(OUT / 'construct_structure_map_E3.csv', index=False)
print(MAP.to_string(index=False))
odd = MAP[MAP.chain != 'A']
if len(odd):
    print('\nchain chosen is not A for:')
    print(odd[['construct', 'entry', 'chain', 'len_chain']].to_string(index=False))
    print('  (1A0N is a complex: chain A is the 14-residue peptide, chain B the SH3 domain)')
cov = (MAP.mapped / MAP.len_construct)
print(f'\nposition coverage: median {cov.median():.1%}, worst {cov.min():.1%} ({MAP.construct[cov.idxmin()]})')
assert (MAP.identity_on_mapped > 0.9).all(), 'a construct does not match its structure chain'

## 3. SaProtΔG and ESM3ΔG

Base weights, three seeds, averaged (Part 6.2). `ddg_scanning=True` returns a full saturation scan
from one structure.

**Sign, fixed by the source and not by the data.** In `ESM3dG.py` the scan is
`dG_mut − dG_wt` and the scale runs −1 → 5 kcal/mol with **higher = more stable**, so their ΔΔG is
**negative = destabilising** — the opposite of ours. We negate. The raw correlation is printed
beside the aligned one so it is visible that the alignment was not chosen to fit.

**Reduction.** `ESM3dG_predict` turns a per-residue ΔG into a scalar with `sum / valid_len`, so a
scan entry is reduced the same way — the mean over the last axis, not the sum.

In [ ]:
EXP = 1                 # 1 = base + measured constructs (MAIN). 2 augmented, 3 AlphaFold, 4 per-seed
RUN_ESM3 = bool(os.environ.get('HF_TOKEN'))    # ESM3 is gated; see the header
VERSION = 'augmented' if EXP == 2 else 'base'
print(f'Experiment {EXP}: version={VERSION}  ESM3={"yes" if RUN_ESM3 else "NOT RUN (no HF_TOKEN)"}')

REPO = Path('/content/absolute-stability-predictor')
if not REPO.exists():
    !git clone -q https://github.com/yehlincho/absolute-stability-predictor {REPO}
    !cd {REPO} && pip -q install -e .
if RUN_ESM3:
    !pip -q install git+https://github.com/evolutionaryScale/esm.git

# SaProtdG parses structures with gemmi, which the package does not pull in
try:
    import gemmi
except ImportError:
    !pip -q install gemmi

# peft refuses to apply LoRA when an old torchao is importable:
#   ImportError: Found an incompatible version of torchao. Found version 0.10.0 ...
# Colab ships 0.10.0 and nothing here needs it, so remove it rather than fight the pin.
try:
    import torchao, importlib.metadata as _md
    v = _md.version('torchao')
    if tuple(int(x) for x in v.split('.')[:2]) < (0, 16):
        print(f'removing torchao {v}: it blocks LoRA and nothing here uses it')
        !pip -q uninstall -y torchao
        import sys as _sys
        for m in [k for k in _sys.modules if k.startswith('torchao')]: del _sys.modules[m]
except ImportError:
    pass

import sys; sys.path.insert(0, str(REPO))

def ckpts(model, version, seed):
    stem = 'ESM3dG' if model == 'ESM3dG' else 'SaProtdG'
    d = REPO / ('esm3dg_weights' if model == 'ESM3dG' else 'saprotdg_weights')
    tag = '_augmented' if version == 'augmented' else ''
    return d / f'{stem}_weights{tag}_{seed}_lora.ckpt'

# The 12 LoRA checkpoints are fetched SEPARATELY from the clone. Gating them on the clone
# means an interrupted first attempt leaves a repo with no weights and every later run
# skips the download - which is exactly what happened.
NEED = [ckpts(m, v, s) for m in ('SaProtdG', 'ESM3dG')
        for v in ('base', 'augmented') for s in (1, 2, 3)]
missing = [c for c in NEED if not c.exists()]

# Drive first, if a copy has been uploaded: the 12 checkpoints are 213 MB and a local
# copy removes the dependency on Hugging Face being reachable and un-rate-limited.
MIRROR = next((PROJECT / 'input' / 'external' / n for n in
               ('models/absolute-stability-predictor', 'models', 'stability/models')
               if (PROJECT / 'input' / 'external' / n).exists()), None)
if missing and MIRROR:
    print(f'{len(missing)} missing; trying the Drive mirror at {MIRROR}')
    for c in list(missing):
        src = MIRROR / c.parent.name / c.name
        if src.exists():
            c.parent.mkdir(parents=True, exist_ok=True)
            try: os.symlink(src, c)
            except OSError: shutil.copy(src, c)
    missing = [c for c in NEED if not c.exists()]
    print(f'  {len(NEED) - len(missing)} of {len(NEED)} now present')

if missing:
    print(f'{len(missing)} of {len(NEED)} checkpoints missing - fetching from Hugging Face')
    !pip -q install -U huggingface_hub
    !cd {REPO} && bash download_weights.sh
    missing = [c for c in NEED if not c.exists()]
for d in (REPO / 'saprotdg_weights', REPO / 'esm3dg_weights'):
    print(f'{d.name}:', sorted(f.name for f in d.glob(chr(42) + '.ckpt')) if d.exists() else 'ABSENT')
assert not missing, ('still missing after download_weights.sh: '
                     + ', '.join(c.name for c in missing)
                     + '. The Hugging Face repo is public; if this persists set HF_TOKEN '
                     + 'and re-run, or fetch by hand with: '
                     + 'hf download Yehlin/absolute-stability --local-dir ' + str(REPO))

# verify against the MD5s this project recorded when the weights were first archived
MANIFEST = PROJECT / 'reference' / 'checksums_models.md5'
if MANIFEST.exists():
    want = dict(l.split()[::-1] for l in MANIFEST.read_text().splitlines()
                if l.strip() and not l.startswith('#'))
    bad = []
    for c in NEED:
        k = f'absolute-stability-predictor/{c.parent.name}/{c.name}'
        if k in want:
            got = hashlib.md5(c.read_bytes()).hexdigest()
            if got != want[k]: bad.append((c.name, got, want[k]))
    assert not bad, f'checkpoint MD5 mismatch (truncated download?): {bad}'
    print(f'all {len(NEED)} checkpoints verified against reference/checksums_models.md5')
else:
    print('no checksums_models.md5 on Drive - checkpoints present but unverified')

In [ ]:
import gc, torch

def scan_to_matrix(ddg_scan):
    """(L,20,B,L) -> (L,20); the mean over residues, matching how dG is reduced to a scalar."""
    # ESM3dG_predict returns ddg_scan.cpu(); SaProtdG_predict returns the raw CUDA tensor.
    t = ddg_scan.detach().cpu() if hasattr(ddg_scan, 'detach') else ddg_scan
    a = np.asarray(t, dtype=np.float64)
    assert a.ndim == 4 and a.shape[1] == 20, f'unexpected scan shape {a.shape}'
    return a[:, :, 0, :].mean(axis=-1)

def scanned_aa(model_name, seq):
    """The amino-acid sequence the model actually scanned.

    SaProt works in a structure-aware alphabet: get_struc_seq returns AA and 3Di interleaved,
    two characters per residue. ESM3 returns the plain sequence."""
    return seq[::2] if model_name == 'SaProtdG' else seq

MODELS = {}
def get_model(model_name, version, seed):
    """Load one checkpoint, holding AT MOST ONE model in memory.

    SaProt is 650M and ESM3 1.4B; three seeds resident at once will not fit a T4, which is
    why seeds are the outer loop below and the previous one is freed here."""
    key = (model_name, version, seed)
    if key in MODELS: return MODELS[key]
    for k in list(MODELS):
        del MODELS[k]
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if model_name == 'ESM3dG':
        from ESM3dG import ESM3dG as Load
    else:
        from SaProtdG import SaProtdG as Load
    ck = ckpts(model_name, version, seed)
    assert ck.exists(), f'checkpoint missing: {ck}'
    print(f'  loading {ck.name} ...', flush=True)
    MODELS[key] = Load(str(ck))
    return MODELS[key]

def run_scan(model_name, version, seed, construct):
    """Cached saturation scan. Returns (matrix (L,20), structure sequence)."""
    key = f'{model_name}_{version}_s{seed}_{construct.replace(".", "_")}'
    f = CACHE / f'{key}.npz'
    if f.exists():
        z = np.load(f, allow_pickle=True)
        return z['m'], str(z['seq'])
    m = MAPPING[construct]
    if model_name == 'ESM3dG':
        from ESM3dG import ESM3dG_predict as Predict
    else:
        from SaProtdG import SaProtdG_predict as Predict
    mdl = get_model(model_name, version, seed)
    ddg, _scaled, seq = Predict(mdl, str(m['path']), m['chain'], ddg_scanning=True)
    mat = scan_to_matrix(ddg)
    # The mapping in section 2 was built from OUR parse of the chain. If the model parsed
    # something else, every score would be attached to the wrong residue and the benchmark
    # would be silently wrong - so check it, on every scan, rather than assume.
    aa, ref = scanned_aa(model_name, seq), m['struct_seq']
    if len(aa) != mat.shape[0] or aa != ref:
        raise AssertionError(
            f'{model_name} scanned a different chain than section 2 mapped for {construct}: '
            f'scan {mat.shape[0]} rows, model sequence {len(aa)}, our chain parse {len(ref)}.'
            + chr(10) + f'  model: {aa[:60]}' + chr(10) + f'  ours : {ref[:60]}')
    np.savez_compressed(f, m=mat, seq=np.array(seq, dtype=object))
    return mat, seq

def competitor_predictions(model_name, version, seeds=(1, 2, 3)):
    """One aligned prediction per held-out variant, averaged over seeds."""
    constructs = sorted(H.domain_key.unique())
    per_seed = {}
    for s in seeds:                       # OUTER: one checkpoint in memory at a time
        print(f'{model_name} {version} seed {s}', flush=True)
        for i, construct in enumerate(constructs, 1):
            mat, _ = run_scan(model_name, version, s, construct)
            per_seed[(s, construct)] = mat
            print(f'    [{i:>2}/{len(constructs)}] {construct}', flush=True)
    out = []
    for construct, g in H.groupby('domain_key'):
        c2s = MAPPING[construct]['c2s']
        vals = []
        for s in seeds:
            mat = per_seed[(s, construct)]
            vals.append([np.nan if (c2s.get(int(p)) is None or c2s[int(p)] >= mat.shape[0]
                                    or mu not in AAS)
                         else mat[c2s[int(p)], AAS.index(mu)]
                         for p, mu in zip(g.pos, g.mut_aa)])
        raw = np.nanmean(np.array(vals, float), axis=0)
        out.append(g.assign(raw=raw, pred_comp=-raw))   # negate: their sign is the opposite
    return pd.concat(out, ignore_index=True)

In [ ]:
# Experiment 4 asks how much the seeds vary, so it reports one arm PER seed
ARMS = [(f'_seed{s}', (s,)) for s in (1, 2, 3)] if EXP == 4 else [('', (1, 2, 3))]

PRED = {}
for suffix, seeds in ARMS:
    PRED['SaProtdG' + suffix] = competitor_predictions('SaProtdG', VERSION, seeds)
    print('SaProtdG' + suffix, 'scored',
          int(PRED['SaProtdG' + suffix].pred_comp.notna().sum()), 'of', len(H))
if RUN_ESM3:
    for suffix, seeds in ARMS:
        PRED['ESM3dG' + suffix] = competitor_predictions('ESM3dG', VERSION, seeds)
        print('ESM3dG' + suffix, 'scored',
              int(PRED['ESM3dG' + suffix].pred_comp.notna().sum()), 'of', len(H))
else:
    print('ESM3dG: NOT RUN - recorded as such, the other arms stand (Part 2)')

## 4. ThermoMPNN — descriptive only, on the two clean domains

Part 6.1. 13 of 15 domains are in its published train or val split, so it is run on **FYN and
NPHP1** and carries **no verdict at any n**. The checkpoint is pinned by commit and MD5; if either
differs the cell stops rather than silently benchmarking a different model.

Sign: ThermoMPNN is trained on MegaScale ΔΔG, which A1 negated on load, so it is negated here too —
the same convention chain, written down rather than fitted.

In [ ]:
TMPNN_COMMIT = '370f76ec62bd929f7425e311d8df04a0d094990f'
TMPNN_MD5    = 'da6f7cf27d921e26af832b112b83cebb'
CLEAN_GENES  = ['FYN', 'NPHP1']          # Part 6.1, fixed before any number existed

TM = Path('/content/ThermoMPNN')
if not TM.exists():
    !git clone -q https://github.com/Kuhlman-Lab/ThermoMPNN {TM}
    !cd {TM} && git checkout -q {TMPNN_COMMIT}
ck = TM / 'models' / 'thermoMPNN_default.pt'
got = hashlib.md5(ck.read_bytes()).hexdigest()
assert got == TMPNN_MD5, f'ThermoMPNN checkpoint is not the pinned one ({got})'
head = subprocess.run(['git', '-C', str(TM), 'rev-parse', 'HEAD'],
                      capture_output=True, text=True).stdout.strip()
assert head == TMPNN_COMMIT, f'repo is at {head}, not the pinned commit'
print('ThermoMPNN pinned:', head[:12], '| checkpoint MD5 verified')

In [ ]:
# ThermoMPNN needs these and Colab has neither by default
try:
    import pytorch_lightning, omegaconf
except ImportError:
    !pip -q install pytorch_lightning omegaconf

def thermompnn(construct):
    m = MAPPING[construct]
    f = CACHE / f'thermompnn_{construct.replace(".", "_")}.csv'
    ch = m['chain']
    if not f.exists():
        d = Path('/content/tm_out'); d.mkdir(exist_ok=True)
        r = subprocess.run(['python', str(TM / 'analysis' / 'custom_inference.py'),
                            '--pdb', str(m['path']), '--chain', ch,
                            '--model_path', str(ck), '--out_dir', str(d)],
                           capture_output=True, text=True, cwd=str(TM))
        if r.returncode != 0:
            msg = (r.stderr or r.stdout or 'no output')[-2500:]
            raise RuntimeError(f'ThermoMPNN failed on {construct} chain {ch}:' + chr(10) + msg)
        src = next(d.glob('ThermoMPNN_inference_*.csv'))
        shutil.copy(src, f); src.unlink()
    t = pd.read_csv(f)
    # its positions index the parsed chain; find the single offset that makes every
    # wild-type residue agree, exactly as E2 did for the Rocklin mapping
    sq = MAPPING[construct]['struct_seq']
    off = None
    for cand in (0, 1, -1):
        if all(0 <= int(pp) - cand < len(sq) and sq[int(pp) - cand] == w
               for pp, w in zip(t.position, t.wildtype)):
            off = cand; break
    assert off is not None, f'no consistent ThermoMPNN offset for {construct}'
    t['sidx'] = t.position.astype(int) - off
    return t.set_index(['sidx', 'mutation']).ddG_pred.astype(float).to_dict()

# This arm carries NO VERDICT (Part 6.1), so a failure here is recorded and the two arms
# that do carry one still report - exactly as Part 2 already allows for ESM3.
HT = H[H.gene.isin(CLEAN_GENES)]
print(f'ThermoMPNN runs on {HT.gene.nunique()} domains, {len(HT):,} variants '
      f'({HT.domain_key.nunique()} constructs)')
TM_RUN, TM_ERR = True, ''
try:
    rows = []
    for construct, g in HT.groupby('domain_key'):
        d = thermompnn(construct)
        c2s = MAPPING[construct]['c2s']
        raw = [d.get((c2s.get(int(pp), -1), mu), np.nan) for pp, mu in zip(g.pos, g.mut_aa)]
        rows.append(g.assign(raw=raw, pred_comp=-np.asarray(raw, float)))
        print(f'  {construct}: {int(np.isfinite(raw).sum())}/{len(g)} scored', flush=True)
    PRED['ThermoMPNN'] = pd.concat(rows, ignore_index=True)
    print('ThermoMPNN scored', int(PRED['ThermoMPNN'].pred_comp.notna().sum()), 'of', len(HT))
except Exception as e:
    TM_RUN, TM_ERR = False, str(e)
    print('ThermoMPNN NOT RUN - it carries no verdict, so this does not block E3.')
    print('reason:', TM_ERR[:2000])

## 5. The comparison

Part 1: per-domain Spearman, median across domains, paired Wilcoxon of the 15 per-domain
correlations against ours. **A variant a competitor cannot score is dropped for everyone**, and the
count dropped is reported.

In [ ]:
def per_domain(df, col='pred_comp'):
    ok = df[df[col].notna()]
    r = (ok.groupby(['group', 'gene'])
           .apply(lambda g: pd.Series({
               'n': len(g),
               'rho_comp': spearmanr(g[col], g.ddG_fold).statistic,
               'rho_ours': spearmanr(g.pred, g.ddG_fold).statistic,
               'rho_comp_raw': spearmanr(g.raw, g.ddG_fold).statistic}),
                  include_groups=False).reset_index())
    return r, len(df) - len(ok)

TABLES, SUMMARY = {}, []
for name, df in PRED.items():
    t, dropped = per_domain(df)
    TABLES[name] = t
    verdict_arm = name.startswith(('SaProtdG', 'ESM3dG'))
    row = dict(competitor=name, domains=len(t), variants_scored=int(t.n.sum()),
               variants_dropped=int(dropped),
               median_rho_comp=float(t.rho_comp.median()),
               median_rho_ours=float(t.rho_ours.median()),
               median_diff=float((t.rho_ours - t.rho_comp).median()),
               median_rho_comp_raw_sign=float(t.rho_comp_raw.median()))
    if not verdict_arm:
        row.update(p=np.nan, verdict='descriptive - no verdict (leakage, Part 6.1)')
    elif len(t) < 5:
        row.update(p=np.nan, verdict='too few domains')
    else:
        p = float(wilcoxon(t.rho_ours, t.rho_comp).pvalue)
        row.update(p=p, verdict=('ours better' if p < ALPHA and row['median_diff'] > 0 else
                                 'competitor better' if p < ALPHA else 'no clear difference'))
    SUMMARY.append(row)
    t.to_csv(OUT / f'per_domain_{name}_E3.csv', index=False)
S = pd.DataFrame(SUMMARY)
S.to_csv(OUT / f'summary_E3_exp{EXP}.csv', index=False)
print(S.to_string(index=False))

## 6. RESULTS_E3.md and metrics_E3.json

In [ ]:
f3 = lambda v: '-' if v != v else f'{v:+.3f}'
met = dict(program='E3', experiment=EXP, version=VERSION,
           date=pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
           preregistration='docs/preregistration/E3.md @ ccd3446', alpha=ALPHA,
           esm3_run=bool(RUN_ESM3), held_out_variants=int(len(H)),
           our_median_rho=float(OURS.rho_ours.median()),
           thermompnn_commit=TMPNN_COMMIT, thermompnn_md5=TMPNN_MD5,
           thermompnn_clean_domains=CLEAN_GENES,
           thermompnn_run=bool(TM_RUN), thermompnn_error=TM_ERR[:500],
           leakage=dict(SaProtdG='0 of 15', ESM3dG='0 of 15', ThermoMPNN='13 of 15'),
           results=S.to_dict('records'))
json.dump(met, open(OUT / f'metrics_E3_exp{EXP}.json', 'w'), indent=1)

L = [f'# E3 - benchmark against current predictors (Experiment {EXP}, {VERSION})', '',
     f'Run {met["date"]}. Machine-written. Plan fixed in advance: '
     '`docs/preregistration/E3.md` @ ccd3446.', '',
     f'15 held-out SH3 domains, {len(H):,} variants. Our numbers are B1 frozen and were '
     f're-derived from it here (median rho {OURS.rho_ours.median():+.3f}).', '',
     '## Leakage audit, reported as a result', '',
     '| competitor | held-out domains in its published training data | role |',
     '|---|---|---|',
     '| SaProtΔG | 0 of 15 | verdict |',
     '| ESM3ΔG | 0 of 15 | verdict |',
     '| ThermoMPNN | **13 of 15** (12 train, 1 val) | descriptive only, 2 clean domains |', '',
     'A MegaScale-derived benchmark cannot compare MegaScale-trained predictors without this '
     'check. Presence or absence is established in the *published* split, not in whatever the '
     'released weights saw.', '',
     '## Comparison', '',
     '| competitor | domains | scored | dropped | median rho | ours | median diff | p | verdict |',
     '|---|---|---|---|---|---|---|---|---|']
for r in SUMMARY:
    L.append(f'| {r["competitor"]} | {r["domains"]} | {r["variants_scored"]:,} | '
             f'{r["variants_dropped"]:,} | {f3(r["median_rho_comp"])} | {f3(r["median_rho_ours"])} | '
             f'{f3(r["median_diff"])} | ' + ('-' if r['p'] != r['p'] else f'{r["p"]:.3g}') +
             f' | **{r["verdict"]}** |')
L += ['', f'Significance: p < {ALPHA} (0.05 across the two competitors carrying a verdict). '
      'Positive median diff means our model correlates better.', '',
      'Signs were aligned by each source documented convention, not by which direction '
      'correlated better. The unaligned median correlations were ' +
      ', '.join(f'{r["competitor"]} {f3(r["median_rho_comp_raw_sign"])}' for r in SUMMARY) + '.', '',
      '## Read with', '',
      '- 15 domains detects large gaps, not small ones - stated in the plan before any result.',
      '- SaProtΔG and ESM3ΔG receive a 3D structure; our model never did.',
      f'- ESM3ΔG {"ran" if RUN_ESM3 else "was NOT RUN (gated model, no token); the other arms stand"}.',
      '- ThermoMPNN carries no verdict at any n. Its two clean domains are FYN and NPHP1.'
      + ('' if TM_RUN else ' **It did not run**; the reason is in metrics_E3.json.')]
(OUT / f'RESULTS_E3_exp{EXP}.md').write_text('\n'.join(L))
print('\n'.join(L))

In [ ]:
print('written to', OUT)
for p in sorted(OUT.iterdir()):
    if p.is_file(): print(f'  {p.stat().st_size:>9,} B  {p.name}')
print(f'\ncached scans: {len(list(CACHE.glob("*")))} files - re-running resumes from here')